# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library. We examine dataset structure, extract records by `@id`, perform preprocessing, and visualize data for further analysis.

### Dataset Source
The dataset is described via a [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).


In [ ]:
# Ensure required libraries are installed
!pip install mlcroissant pandas matplotlib seaborn --quiet

## 1. Data Loading
Load metadata and examine available record sets with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load metadata and dataset contents
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

# Print dataset overview
print(f"Dataset Name: {meta.name}\n")
print(f"Description: {meta.description}\n")
print(f"Version: {getattr(meta, 'version', 'Unknown')}")
print(f"Published: {getattr(meta, 'datePublished', 'Unknown')}")
print(f"License: {getattr(meta, 'license', 'Unknown')}")

## 2. Data Overview
List available record sets, their fields, and respective `@id`s. All subsequent data access will be performed by `@id`.

In [ ]:
# Explore record sets

record_sets = dataset.record_sets
print(f"Available Record Sets (referenced by @id):\n")
for rs in record_sets:
    print(f" - {rs.id}: {getattr(rs, 'name', '')}")
    if hasattr(rs, 'fields') and rs.fields:
        for field in rs.fields:
            print(f"     field @id: {field.id}" + (f" name: {getattr(field, 'name', '')}" if hasattr(field,'name') else ''))
    if hasattr(rs, 'columns') and rs.columns:
        for col in rs.columns:
            print(f"     column @id: {col.id}" + (f" name: {getattr(col, 'name', '')}" if hasattr(col,'name') else ''))

## 3. Data Extraction
Load data from each record set of interest into a DataFrame. Use the record set and field `@id` values from the previous overview.

In [ ]:
# For this dataset, the main record set appears as a major table of protein analysis record.
# Update these IDs based on the overview from above. We will use the first tabular record set (by @id) for analysis.

# Collect all tabular record set @ids
main_record_set_id = None
tabular_record_sets = []

for rs in record_sets:
    # RecordSet objects in mlcroissant should have id (the @id) and possibly a 'record_set_type' or columns/fields
    if hasattr(rs, 'fields') and rs.fields:
        tabular_record_sets.append(rs.id)
if tabular_record_sets:
    main_record_set_id = tabular_record_sets[0]

print(f"Selected primary record set: {main_record_set_id}\n")

# Load all available tabular record sets into pandas DataFrames
dataframes = {}
for recset_id in tabular_record_sets:
    records = list(dataset.records(record_set=recset_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
        print(f"Loaded {len(df)} records for {recset_id}")

if main_record_set_id is not None and main_record_set_id in dataframes:
    print("\nDataFrame columns (@id):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We'll filter, normalize, and group by fields. All field references use their `@id` strings.


In [ ]:
# If you are unsure which numeric fields are present, uncomment this block to inspect column names and types:
# df = dataframes[main_record_set_id]
# print(df.dtypes)  # Use this to select a relevant numeric field

# For demonstration, insert the appropriate numeric field @id as seen previously. Example: 'cr:peptideCount', 'cr:molecularWeight' (MW), etc.
# You can change these IDs according to your data structure:
numeric_field_id = None
group_field_id = None

df = dataframes[main_record_set_id]
print("Columns available for EDA:\n", df.columns.tolist())
# Pick a likely numeric field:
for col in df.columns:
    if 'count' in col.lower() or 'weight' in col.lower() or 'abundance' in col.lower() or 'percentage' in col.lower() or 'MW' in col:
        numeric_field_id = col
        break
# Pick a grouping field, e.g., description, accession, or experiment/sample id
for col in df.columns:
    if 'description' in col.lower() or 'sample' in col.lower() or 'experiment' in col.lower():
        group_field_id = col
        break

print(f"\nUsing numeric field (@id): {numeric_field_id}")
if group_field_id is not None:
    print(f"Using group field (@id): {group_field_id}")
else:
    print("No clear group field detected.")

# Clean and convert numeric field
df_filtered = df.copy()
# Convert to numeric, coerce invalid values to NaN, drop NaNs
df_filtered[numeric_field_id] = pd.to_numeric(df_filtered[numeric_field_id], errors='coerce')
df_filtered = df_filtered.dropna(subset=[numeric_field_id])

# Filtering - keep only rows with field > 10
threshold = 10
filtered = df_filtered[df_filtered[numeric_field_id] > threshold].copy()
print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
print(filtered[[numeric_field_id]].head())

# Normalization (Z-score)
filtered[f"{numeric_field_id}_normalized"] = (filtered[numeric_field_id] - filtered[numeric_field_id].mean()) / filtered[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by the detected group field and show averages
if group_field_id and group_field_id in filtered.columns:
    grouped = filtered.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data (mean {numeric_field_id}) by {group_field_id}:")
    print(grouped.head())

## 5. Visualization
Visualizing the numeric field's distribution and (optional) its grouping.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of values
plt.figure(figsize=(8, 4))
sns.histplot(filtered[numeric_field_id], bins=30, kde=True, color='skyblue')
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id} (filtered > {threshold})")
plt.tight_layout()
plt.show()

# If grouped, plot top categories
if group_field_id and group_field_id in filtered:
    grouped_sorted = filtered.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(10)
    plt.figure(figsize=(10,6))
    sns.barplot(x=grouped_sorted.values, y=grouped_sorted.index, palette="viridis")
    plt.xlabel(f"Mean {numeric_field_id}")
    plt.ylabel(group_field_id)
    plt.title(f"Top 10 {group_field_id} by mean {numeric_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
* Using the Croissant schema, we loaded and explored the FAIR^2 dataset using `mlcroissant`.
* We listed all record sets and fields by their `@id`s, filtered and normalized a numeric field, and visualized its distribution as well as group-wise means.
* This approach can be extended to more specialized analyses by substituting relevant field `@id`s and performing deeper domain-specific transformations.
